# LAS CMS — Phase 3.2: Data Cleaning (8 Tasks)
---

In [1]:
import pandas as pd, numpy as np, re, json
from pathlib import Path
from datetime import datetime

DATA_DIR = Path("../outputs/anonymized")
OUTPUT_DIR = Path("../outputs/cleaned")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cleaning_log = []
def log(t, d): cleaning_log.append({'task':t,'details':d}); print(f"  → {d}")

programs = pd.read_csv(DATA_DIR/"programs_anonymized.csv")
print(f"✓ programs: {len(programs):,} rows, {len(programs.columns)} cols")

hearings = pd.DataFrame()
try: hearings = pd.read_csv(DATA_DIR/"hearings_anonymized.csv"); print(f"✓ hearings: {len(hearings):,}")
except: print("⚠ hearings not found")

✓ programs: 3,886 rows, 60 cols
✓ hearings: 13,740


## Task 1: Court Level — `levelOfCourt`

In [2]:
if 'levelOfCourt' in programs.columns:
    print("BEFORE:", dict(programs['levelOfCourt'].value_counts()))
    
    # Known lawyer names that leaked into court field (CMS data entry error)
    known_bad = ['naveed channa','syeda bushra','muhammad tasleem','mehwish naz',
                 'danial ali','barkat ali','abida bibi','noman maqbool',
                 'munawar abbas','amanullah wassan','ayaz ahmed']
    
    def fix_court(v):
        if pd.isna(v) or str(v).strip() == '': return 'UNKNOWN'
        vl = str(v).strip().lower()
        
        # Flag lawyer names and garbage as UNKNOWN
        if vl in known_bad: return 'UNKNOWN'
        if vl.replace('.','').isdigit(): return 'UNKNOWN'  # e.g. "3118"
        
        # Standardize court names
        if 'family' in vl: return 'Family Court'
        if 'session' in vl: return 'Sessions Court'
        if 'district' in vl: return 'District Court'
        if 'high' in vl: return 'High Court'
        if 'supreme' in vl: return 'Supreme Court'
        if 'civil' in vl or 'magistrate' in vl: return 'Civil/Magistrate Court'
        return 'UNKNOWN'
    
    programs['levelOfCourt'] = programs['levelOfCourt'].apply(fix_court)
    print("\nAFTER:", dict(programs['levelOfCourt'].value_counts()))
    log("T1", f"→ {programs['levelOfCourt'].nunique()} values")

BEFORE: {'Family': 874, 'Civil & Judicial Magistrate': 399, 'Additional District & Sessions': 391, 'family court': 238, 'District & Sessions': 114, 'district & sessions court': 108, 'Senior Civil/Assistant Sessions': 106, 'judicial magistrate': 34, 'Naveed Channa': 31, 'High Court ': 24, 'civil judge/ senior civil judge': 21, 'High Court': 21, 'high court / supreme court': 18, 'District & Sessions Court': 13, 'Judicial Magistrate': 11, 'Family Court': 9, 'Syeda Bushra': 8, 'Muhammad Tasleem': 6, 'Mehwish Naz': 5, 'Danial Ali': 4, 'High Court / Supreme Court': 3, 'Civil Judge/ Senior Civil Judge': 3, 'Abida Bibi': 2, 'Barkat Ali': 2, '3118': 1, 'Supreme Court of Pakistan': 1, 'Additional Session Judge': 1, 'Noman Maqbool': 1, 'Munawar Abbas': 1, 'Amanullah Wassan': 1, 'Ayaz Ahmed': 1}

AFTER: {'UNKNOWN': 1497, 'Family Court': 1121, 'Sessions Court': 733, 'Civil/Magistrate Court': 468, 'High Court': 66, 'Supreme Court': 1}
  → → 6 values


## Task 2: Parse Dates

In [3]:
for col in ['interviewDate','caseFileDate','caseDisposalDate','approvalDate','vakalatnamaSubmissionDate','created_at','updated_at']:
    if col in programs.columns:
        programs[f'{col}_parsed'] = pd.to_datetime(programs[col], errors='coerce')
        ok = programs[f'{col}_parsed'].notna().sum()
        print(f"  {col}: {ok:,} parsed")
if not hearings.empty:
    for col in ['date','nextHearing','created_at']:
        if col in hearings.columns:
            hearings[f'{col}_parsed'] = pd.to_datetime(hearings[col], errors='coerce')
            print(f"  hearings.{col}: {hearings[f'{col}_parsed'].notna().sum():,}")

  interviewDate: 3,484 parsed
  caseFileDate: 2,336 parsed
  caseDisposalDate: 1,940 parsed
  approvalDate: 2,623 parsed
  vakalatnamaSubmissionDate: 2,309 parsed
  created_at: 1,674 parsed
  updated_at: 1,816 parsed
  hearings.date: 6
  hearings.nextHearing: 13,487
  hearings.created_at: 13,624


## Task 3: Standardize Decisions

In [4]:
if 'caseDecision' in programs.columns:
    def fix_dec(v):
        if pd.isna(v) or str(v).strip()=='': return 'PENDING'
        vl = str(v).strip().lower()
        if any(k in vl for k in ['favour','favor','won','granted']): return 'IN_FAVOUR'
        if any(k in vl for k in ['against','dismiss','lost']): return 'AGAINST'
        if any(k in vl for k in ['comprom','settl']): return 'COMPROMISE'
        if 'withdraw' in vl: return 'WITHDRAWN'
        if 'transfer' in vl: return 'TRANSFERRED'
        return str(v).strip()
    programs['caseDecision_clean'] = programs['caseDecision'].apply(fix_dec)
    print(dict(programs['caseDecision_clean'].value_counts()))

{'IN_FAVOUR': 1627, 'PENDING': 1527, 'In Progress': 358, 'AGAINST': 339, 'Missing': 22, 'TRANSFERRED': 12, 'Disposed of': 1}


## Task 4: Flag Fake CNICs

In [5]:
if 'cnic' in programs.columns:
    programs['cnic_quality'] = 'VALID'
    programs.loc[programs['cnic']=='FAKE_PLACEHOLDER', 'cnic_quality'] = 'FAKE'
    programs.loc[programs['cnic'].isna()|(programs['cnic']==''), 'cnic_quality'] = 'MISSING'
    print(dict(programs['cnic_quality'].value_counts()))

{'VALID': 2048, 'FAKE': 925, 'MISSING': 913}


## Task 5: Flag Missing Outcomes

In [6]:
if 'caseDecision_clean' in programs.columns:
    programs['outcome_quality'] = 'HAS_OUTCOME'
    programs.loc[programs['caseDecision_clean']=='PENDING', 'outcome_quality'] = 'MISSING_OUTCOME'
    if 'currentCaseStatus' in programs.columns:
        m = programs['currentCaseStatus'].astype(str).str.contains('Disposed', case=False, na=False) & (programs['caseDecision_clean']=='PENDING')
        programs.loc[m, 'outcome_quality'] = 'DATA_ERROR'
        print(f"Data errors (Disposed+no decision): {m.sum()}")
    print(dict(programs['outcome_quality'].value_counts()))

Data errors (Disposed+no decision): 15
{'HAS_OUTCOME': 2359, 'MISSING_OUTCOME': 1512, 'DATA_ERROR': 15}


## Task 6: Standardize Nature of Case

In [7]:
if 'natureOfCase' in programs.columns:
    print("BEFORE:", dict(programs['natureOfCase'].value_counts()))
    
    def fix_nat(v):
        if pd.isna(v) or str(v).strip() == '': return 'UNSPECIFIED'
        vl = str(v).strip().lower()
        
        # Flag years that leaked into this field (CMS data entry error)
        if vl.isdigit() and len(vl) == 4: return 'UNSPECIFIED'
        
        # Map actual values found in this DB
        if 'family' in vl: return 'Family'
        if 'criminal' in vl or 'cr.' in vl: return 'Criminal'
        if 'civil' in vl: return 'Civil'
        if 'constitutional' in vl: return 'Constitutional'
        
        # Also handle specific case types if they exist
        if 'domestic' in vl or 'violence' in vl: return 'Family'
        if 'divorce' in vl or 'khula' in vl: return 'Family'
        if 'custody' in vl or 'guardian' in vl: return 'Family'
        if 'maintenance' in vl or 'nafqa' in vl: return 'Family'
        if 'inheritance' in vl or 'property' in vl: return 'Civil'
        if 'rape' in vl or 'sexual' in vl: return 'Criminal'
        if 'harassment' in vl: return 'Criminal'
        if 'murder' in vl: return 'Criminal'
        if 'fraud' in vl: return 'Criminal'
        return str(v).strip().title()
    
    programs['natureOfCase_clean'] = programs['natureOfCase'].apply(fix_nat)
    print("\nAFTER:", dict(programs['natureOfCase_clean'].value_counts()))
    log("T6", f"→ {programs['natureOfCase_clean'].nunique()} categories")

BEFORE: {'Family': 1487, 'Criminal': 726, '2024': 204, 'Civil': 176, '2025': 110, '2026': 38, 'Constitutional Petition': 23, 'Cr. Misc': 1}

AFTER: {'Family': 1487, 'UNSPECIFIED': 1473, 'Criminal': 727, 'Civil': 176, 'Constitutional': 23}
  → → 5 categories


## Task 7: Duplicates

In [8]:
b = len(programs); programs = programs.drop_duplicates()
dc = [c for c in ['clientName','natureOfCase','caseFileDate'] if c in programs.columns]
if len(dc)>=2:
    programs['is_possible_duplicate'] = programs.duplicated(subset=dc, keep=False)
    print(f"Flagged: {programs['is_possible_duplicate'].sum()}")
print(f"Before: {b:,} → After: {len(programs):,} (removed {b-len(programs):,})")

Flagged: 1054
Before: 3,886 → After: 3,858 (removed 28)


## Task 8: Impossible Dates

In [9]:
if 'caseFileDate_parsed' in programs.columns and 'caseDisposalDate_parsed' in programs.columns:
    dur = (programs['caseDisposalDate_parsed'] - programs['caseFileDate_parsed']).dt.days
    programs['date_quality'] = 'OK'
    programs.loc[dur<0, 'date_quality'] = 'NEGATIVE_DURATION'
    programs.loc[dur>3650, 'date_quality'] = 'EXTREME_DURATION'
    for c in ['caseFileDate_parsed','caseDisposalDate_parsed']:
        if c in programs.columns: programs.loc[programs[c]>pd.Timestamp.now(), 'date_quality'] = 'FUTURE_DATE'
    print(dict(programs['date_quality'].value_counts()))

{'OK': 3812, 'NEGATIVE_DURATION': 42, 'FUTURE_DATE': 3, 'EXTREME_DURATION': 1}


## Save

In [10]:
temp = [c for c in programs.columns if c.startswith('_')]
programs.drop(columns=temp, inplace=True, errors='ignore')

programs.to_csv(OUTPUT_DIR/"programs_cleaned.csv", index=False)
print(f"✓ programs_cleaned.csv ({len(programs):,} rows, {len(programs.columns)} cols)")
if not hearings.empty:
    hearings.to_csv(OUTPUT_DIR/"hearings_cleaned.csv", index=False)
    print(f"✓ hearings_cleaned.csv ({len(hearings):,})")

pd.DataFrame(cleaning_log).to_csv(Path("../outputs")/"cleaning_log.csv", index=False)

# Re-score
s = min(10, len(programs)/500*10)
for c in ['caseDecision_clean','natureOfCase_clean','levelOfCourt','districtName']:
    if c in programs.columns: s += programs[c].notna().sum()/len(programs)*5
if 'caseFacts' in programs.columns:
    s += (programs['caseFacts'].fillna('').astype(str).str.len()>100).sum()/len(programs)*20
if 'caseDecision_clean' in programs.columns:
    s += (programs['caseDecision_clean']!='PENDING').sum()/len(programs)*15
s += 12 + 8
print(f"\n🎯 AI Readiness: 62 → {s:.0f}/100")
print("\n✅ Next: 04_rag_preparation.ipynb")

✓ programs_cleaned.csv (3,858 rows, 73 cols)
✓ hearings_cleaned.csv (13,740)

🎯 AI Readiness: 62 → 69/100

✅ Next: 04_rag_preparation.ipynb
